# Video → 3D (VGGT on Colab)

This notebook takes **one video** and creates **two outputs of that same video**:

| File | Meaning |
|------|---------|
| `scene.glb` | Interactive 3D of the space in your video |
| `flythrough.mp4` | Moving camera path through that 3D |

Then upload both files in your Streamlit app for the job.

---

## How to run (do this in order)

1. **Runtime → Change runtime type → T4 GPU** (or any GPU)
2. Run **Step 0** (install)
3. Click **Runtime → Restart session**
4. Run **Step 1 → Step 7** in order (do **not** re-run Step 0 after restart unless install failed)
5. Download `scene.glb` and `flythrough.mp4`


## Step 0 — Install packages

Run once, then **restart the session**.


In [ ]:
import os
import sys
import shutil

SRC = "/content/vggt_src"

# Old notebooks cloned into /content/vggt and broke imports. Remove that trap.
if os.path.isdir("/content/vggt") and not os.path.isfile("/content/vggt/models/vggt.py"):
    print("Removing shadowing folder /content/vggt ...")
    shutil.rmtree("/content/vggt")

if not os.path.isdir(os.path.join(SRC, "vggt", "models")):
    print("Cloning facebookresearch/vggt ...")
    !git clone --depth 1 https://github.com/facebookresearch/vggt.git {SRC}
else:
    print("VGGT source already present:", SRC)

# Clean NumPy binary (avoids: numpy.dtype size changed)
!{sys.executable} -m pip -q uninstall -y numpy
!{sys.executable} -m pip -q install "numpy==1.26.4"

!{sys.executable} -m pip -q install \
  Pillow huggingface_hub einops safetensors \
  opencv-python-headless trimesh matplotlib scipy tqdm \
  imageio imageio-ffmpeg

!{sys.executable} -m pip -q install -e {SRC}

print()
print("Install finished.")
print("Now click: Runtime -> Restart session")
print("After restart, start from Step 1 (skip Step 0).")


## Step 1 — Import VGGT (AFTER restart)


In [ ]:
import sys

SRC = "/content/vggt_src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import numpy as np
import torch

print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable GPU: Runtime -> Change runtime type -> T4 GPU, then Restart and rerun Step 1."

from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map

print("VGGT imports OK")


## Step 2 — Upload your video

Upload the **same video** you used in Streamlit (or any short indoor clip).


In [ ]:
from google.colab import files

uploaded = files.upload()
assert uploaded, "Please upload one video file."

VIDEO_PATH = list(uploaded.keys())[0]
print("Using video:", VIDEO_PATH)


## Step 3 — Extract frames from the video

About **1 frame/sec**, max **40 frames** (better for free Colab memory).


In [ ]:
import cv2
from pathlib import Path

frames_dir = Path("/content/frames")
frames_dir.mkdir(exist_ok=True)
for old in frames_dir.glob("*"):
    old.unlink()

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
interval = max(1, int(round(fps)))  # ~1 FPS
max_frames = 40

idx = 0
saved = 0
while saved < max_frames:
    ok, frame = cap.read()
    if not ok:
        break
    if idx % interval == 0:
        out = frames_dir / f"{saved:06d}.jpg"
        cv2.imwrite(str(out), frame)
        saved += 1
    idx += 1
cap.release()

image_names = sorted(str(p) for p in frames_dir.glob("*.jpg"))
print(f"Extracted {len(image_names)} frames from {VIDEO_PATH}")
assert len(image_names) >= 2, "Need at least 2 frames. Try a longer / clearer clip."
print("Examples:", image_names[:3])


## Step 4 — Run VGGT reconstruction on those frames


In [ ]:
import sys
import numpy as np
import torch

SRC = "/content/vggt_src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map


def to_numpy(x):
    # Torch -> NumPy. Squeeze batch dim only when it is size 1.
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    x = np.asarray(x)
    if x.ndim >= 1 and x.shape[0] == 1:
        x = x.reshape(x.shape[1:])
    return x


device = "cuda"
model = VGGT.from_pretrained("facebook/VGGT-1B").to(device)
model.eval()

images = load_and_preprocess_images(image_names).to(device)  # (S, 3, H, W)
print("images shape:", tuple(images.shape))

dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
with torch.no_grad():
    with torch.cuda.amp.autocast(dtype=dtype):
        predictions = model(images)

# pose_enc must be (B, S, 9) for the official converter
pose = predictions["pose_enc"]
print("pose_enc raw shape:", tuple(pose.shape))
if pose.ndim == 2:  # (S, 9) -> (1, S, 9)
    pose = pose.unsqueeze(0)
predictions["pose_enc"] = pose

extrinsic, intrinsic = pose_encoding_to_extri_intri(pose, images.shape[-2:])
predictions["extrinsic"] = extrinsic
predictions["intrinsic"] = intrinsic
predictions["images"] = images

for key, value in list(predictions.items()):
    if isinstance(value, torch.Tensor):
        predictions[key] = to_numpy(value)

predictions["pose_enc_list"] = None
predictions["world_points_from_depth"] = unproject_depth_map_to_point_map(
    predictions["depth"],
    predictions["extrinsic"],
    predictions["intrinsic"],
)

print("extrinsic shape:", predictions["extrinsic"].shape)
print("depth shape:", predictions["depth"].shape)
print("Reconstruction done for this video")


## Step 5 — Export `scene.glb` (interactive 3D)


In [ ]:
import sys

SRC = "/content/vggt_src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from visual_util import predictions_to_glb

scene = predictions_to_glb(
    predictions,
    conf_thres=50.0,
    filter_by_frames="all",
    show_cam=True,
    prediction_mode="Predicted Pointmap",
)

GLB_PATH = "/content/scene.glb"
scene.export(GLB_PATH)
print("Wrote", GLB_PATH)


## Step 6 — Export `flythrough.mp4` (moving path through the same 3D)


In [ ]:
import numpy as np
import cv2

# Point cloud from THIS video's reconstruction
if "world_points" in predictions:
    pts = np.asarray(predictions["world_points"]).reshape(-1, 3)
    conf = np.asarray(predictions.get("world_points_conf", np.ones(len(pts)))).reshape(-1)
else:
    pts = np.asarray(predictions["world_points_from_depth"]).reshape(-1, 3)
    conf = np.asarray(predictions.get("depth_conf", np.ones(len(pts)))).reshape(-1)

keep = conf >= np.percentile(conf, 60)
pts = pts[keep]
if len(pts) > 120000:
    idx = np.random.default_rng(0).choice(len(pts), 120000, replace=False)
    pts = pts[idx]

extrinsics = np.asarray(predictions["extrinsic"])  # (S, 3, 4)
assert extrinsics.ndim == 3 and extrinsics.shape[-2:] == (3, 4), extrinsics.shape

center = np.median(pts, axis=0)
scale = np.percentile(np.linalg.norm(pts - center, axis=1), 90) + 1e-6
pts_n = (pts - center) / scale

W, H = 720, 480
FT_PATH = "/content/flythrough.mp4"
writer = cv2.VideoWriter(FT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), 12.0, (W, H))


def render_frame(E):
    # Project the point cloud with a world-to-camera 3x4 matrix.
    R, t = E[:, :3], E[:, 3]
    tn = (R @ center + t) / scale
    Xc = (R @ pts_n.T).T + tn
    z = Xc[:, 2]
    valid = z > 0.05

    frame = np.full((H, W, 3), 18, dtype=np.uint8)
    if int(valid.sum()) < 50:
        return frame

    u = Xc[valid, 0] / z[valid]
    v = Xc[valid, 1] / z[valid]
    u0, u1 = np.percentile(u, [5, 95])
    v0, v1 = np.percentile(v, [5, 95])
    if u1 - u0 < 1e-3:
        u1 = u0 + 1e-3
    if v1 - v0 < 1e-3:
        v1 = v0 + 1e-3

    px = np.clip(((u - u0) / (u1 - u0) * (W - 1)).astype(np.int32), 0, W - 1)
    py = np.clip(((v - v0) / (v1 - v0) * (H - 1)).astype(np.int32), 0, H - 1)

    depth = z[valid]
    col = (255 * (1.0 - (depth - depth.min()) / (np.ptp(depth) + 1e-6))).astype(np.uint8)
    frame[py, px, 0] = col
    frame[py, px, 1] = col
    frame[py, px, 2] = np.clip(col.astype(np.int32) + 40, 0, 255).astype(np.uint8)
    return frame


# Smooth path: interpolate between consecutive camera poses
for i in range(len(extrinsics) - 1):
    for a in np.linspace(0.0, 1.0, 4, endpoint=False):
        E = (1.0 - a) * extrinsics[i] + a * extrinsics[i + 1]
        writer.write(render_frame(E))
writer.write(render_frame(extrinsics[-1]))
writer.release()

print("Wrote", FT_PATH)


## Step 7 — Download both files

In Streamlit, attach `scene.glb` and `flythrough.mp4`.


In [ ]:
from google.colab import files
from pathlib import Path

assert Path("/content/scene.glb").is_file(), "scene.glb missing — rerun Step 5"
assert Path("/content/flythrough.mp4").is_file(), "flythrough.mp4 missing — rerun Step 6"

files.download("/content/scene.glb")
files.download("/content/flythrough.mp4")
print("Downloads started. Upload both into your Streamlit job.")
